In [1]:
!pip install -q transformers scikit-learn pandas pillow joblib

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
# ============================================================
# CLIP ALTERNATIVE TEXT CLASSIFICATION
# CLIP SIMILARITY THRESHOLD VS LOGISTIC REGRESSION
# ============================================================

# Libraries
import os
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

from PIL import Image
from pathlib import Path

from transformers import CLIPProcessor, CLIPModel

from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve
)

import joblib

In [4]:
# ============================================================
# 1. CONFIGURATION
# ============================================================

SEED = 42

DATASET_DIR = (
    "/content/drive/MyDrive/EEE6778_Semester_Project/"
)

CSV_PATH = os.path.join(
    DATASET_DIR,
    "doclaynet_synthetic_dataset/synthetic_alt_text_dataset.csv"
)

IMAGE_DIR = os.path.join(DATASET_DIR, "images")

OUTPUT_DIR = os.path.join(
    DATASET_DIR,
    "clip_training_results"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"

BATCH_SIZE = 16

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)
print("Dataset directory:", DATASET_DIR)
print("Output directory:", OUTPUT_DIR)

Device: cpu
Dataset directory: /content/drive/MyDrive/EEE6778_Semester_Project/
Output directory: /content/drive/MyDrive/EEE6778_Semester_Project/clip_training_results


In [5]:
# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [6]:
# ============================================================
# 3. LOAD DATASET
# ============================================================

df = pd.read_csv(CSV_PATH)

print("\nOriginal dataset shape:", df.shape)

required_columns = [
    "document_id",
    "figure_id",
    "image_path",
    "alt_text",
    "alt_text_class"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

# Remove rows with missing text or image paths.
df = df.dropna(
    subset=["image_path", "alt_text", "alt_text_class"]
).copy()

df["alt_text"] = df["alt_text"].astype(str).str.strip()

df = df[df["alt_text"] != ""].copy()



Original dataset shape: (2002, 8)


In [7]:
# ============================================================
# 4. NORMALIZE CLASS LABELS
# ============================================================

def convert_label(value):

    value = str(value).strip().lower()

    label_mapping = {
        "1": 1,
        "0": 0,
        "meaningful": 1,
        "nonmeaningful": 0,
        "non-meaningful": 0,
        "meaningful alt text": 1,
        "nonmeaningful alt text": 0
    }

    if value not in label_mapping:
        raise ValueError(
            f"Unrecognized alt_text_class label: {value}"
        )

    return label_mapping[value]


df["label"] = df["alt_text_class"].apply(convert_label)

print("\nClass distribution:")
print(df["label"].value_counts().sort_index())

if df["label"].nunique() != 2:
    raise ValueError("The dataset must contain both classes.")


Class distribution:
label
0     980
1    1001
Name: count, dtype: int64


In [8]:
# ============================================================
# 5. RESOLVE IMAGE PATHS
# ============================================================

def resolve_image_path(path):

    path = str(path)

    candidates = [
        Path(path),
        Path(DATASET_DIR) / path,
        Path(IMAGE_DIR) / Path(path).name
    ]

    for candidate in candidates:

        if candidate.exists():
            return str(candidate.resolve())

    raise FileNotFoundError(
        f"Could not find image: {path}"
    )


df["resolved_image_path"] = df["image_path"].apply(
    resolve_image_path
)

print("\nNumber of usable records:", len(df))

print(
    "Number of unique images:",
    df["resolved_image_path"].nunique()
)



Number of usable records: 1981
Number of unique images: 1001


In [9]:
# ============================================================
# 6. CREATE GROUP IDENTIFIERS
# ============================================================

# Prefer document-level splitting.
# If there are fewer than three unique document IDs,
# fall back to grouping by image so paired descriptions
# remain together.

if df["document_id"].nunique() >= 3:

    df["split_group"] = df["document_id"].astype(str)

    print("\nSplitting by document_id.")

else:

    df["split_group"] = df["resolved_image_path"]

    print(
        "\nWARNING: Fewer than three unique document IDs."
    )

    print("Splitting by image instead.")


Splitting by document_id.


In [10]:
# ============================================================
# 7. TRAIN / VALIDATION / TEST SPLIT
# ============================================================

# First: 85% training + validation, 15% test.
splitter_test = GroupShuffleSplit(
    n_splits=1,
    test_size=0.15,
    random_state=SEED
)

train_val_idx, test_idx = next(
    splitter_test.split(
        df,
        df["label"],
        groups=df["split_group"]
    )
)

train_val_df = df.iloc[train_val_idx].copy()
test_df = df.iloc[test_idx].copy()

# Second: split the remaining 85% into approximately
# 70% training and 15% validation overall.

splitter_val = GroupShuffleSplit(
    n_splits=1,
    test_size=0.17647,
    random_state=SEED
)

train_idx, val_idx = next(
    splitter_val.split(
        train_val_df,
        train_val_df["label"],
        groups=train_val_df["split_group"]
    )
)

train_df = train_val_df.iloc[train_idx].copy()
val_df = train_val_df.iloc[val_idx].copy()

# Reset indices for convenient indexing.
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("\nDataset split:")
print("------------------------------")

for name, subset in [
    ("Training", train_df),
    ("Validation", val_df),
    ("Testing", test_df)
]:

    print(f"\n{name}:")
    print("Records:", len(subset))
    print("Unique images:", subset["resolved_image_path"].nunique())
    print("Class distribution:")
    print(subset["label"].value_counts().sort_index())

# Verify that no groups overlap.
train_groups = set(train_df["split_group"])
val_groups = set(val_df["split_group"])
test_groups = set(test_df["split_group"])

assert train_groups.isdisjoint(val_groups)
assert train_groups.isdisjoint(test_groups)
assert val_groups.isdisjoint(test_groups)

print("\nGroup leakage check passed.")


# Save split files for reproducibility.
train_df.to_csv(
    os.path.join(OUTPUT_DIR, "train.csv"),
    index=False
)

val_df.to_csv(
    os.path.join(OUTPUT_DIR, "validation.csv"),
    index=False
)

test_df.to_csv(
    os.path.join(OUTPUT_DIR, "test.csv"),
    index=False
)


Dataset split:
------------------------------

Training:
Records: 1431
Unique images: 725
Class distribution:
label
0    706
1    725
Name: count, dtype: int64

Validation:
Records: 321
Unique images: 161
Class distribution:
label
0    160
1    161
Name: count, dtype: int64

Testing:
Records: 229
Unique images: 115
Class distribution:
label
0    114
1    115
Name: count, dtype: int64

Group leakage check passed.


In [11]:
# ============================================================
# 8. LOAD PRETRAINED CLIP
# ============================================================

print("\nLoading CLIP model...")

processor = CLIPProcessor.from_pretrained(
    CLIP_MODEL_NAME
)

clip_model = CLIPModel.from_pretrained(
    CLIP_MODEL_NAME
)

clip_model = clip_model.to(DEVICE)
clip_model.eval()

print("CLIP loaded successfully.")


Loading CLIP model...


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

... (4 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIP loaded successfully.


In [14]:
# ------------------------------------------------------------
# 9 GENERATE EMBEDDINGS FUNCTION
# ------------------------------------------------------------

def generate_embeddings(
    dataframe,
    batch_size,
    model,
    processor,
    device
):

    if len(dataframe) == 0:
        raise ValueError(
            "The supplied dataframe is empty."
        )

    required_columns = ["image_path", "alt_text"]

    for column in required_columns:
        if column not in dataframe.columns:
            raise ValueError(
                f"Missing required column: {column}"
            )

    model.eval()

    all_image_embeddings = []
    all_text_embeddings = []

    # Disable gradient calculations.
    with torch.no_grad():

        for start in tqdm(
            range(0, len(dataframe), batch_size),
            desc="Generating embeddings"
        ):

            batch = dataframe.iloc[
                start:start + batch_size
            ]

            # ------------------------------------------------
            # LOAD IMAGES
            # ------------------------------------------------

            images = []

            for image_path in batch["image_path"]:

                if not os.path.exists(image_path):
                    raise FileNotFoundError(
                        f"Image not found: {image_path}"
                    )

                image = Image.open(
                    image_path
                ).convert("RGB")

                images.append(image)

            # ------------------------------------------------
            # LOAD ALTERNATIVE TEXT
            # ------------------------------------------------

            texts = (
                batch["alt_text"]
                .fillna("")
                .astype(str)
                .tolist()
            )

            # ------------------------------------------------
            # PROCESS IMAGES
            # ------------------------------------------------

            image_inputs = processor(
                images=images,
                return_tensors="pt"
            )

            image_inputs = {
                key: value.to(device)
                for key, value in image_inputs.items()
            }

            # ------------------------------------------------
            # PROCESS TEXT
            # ------------------------------------------------

            text_inputs = processor(
                text=texts,
                return_tensors="pt",
                padding=True,
                truncation=True
            )

            text_inputs = {
                key: value.to(device)
                for key, value in text_inputs.items()
            }

            # ------------------------------------------------
            # EXTRACT CLIP IMAGE EMBEDDINGS
            # ------------------------------------------------

            image_features = model.get_image_features(
                **image_inputs
            )

            # ------------------------------------------------
            # EXTRACT CLIP TEXT EMBEDDINGS
            # ------------------------------------------------

            text_features = model.get_text_features(
                **text_inputs
            )

            # ------------------------------------------------
            # HANDLE TRANSFORMERS OUTPUT TYPES
            # ------------------------------------------------

            # Some Transformers versions return a model
            # output object instead of a direct tensor.

            if not isinstance(image_features, torch.Tensor):

                if hasattr(image_features, "pooler_output"):
                    image_features = image_features.pooler_output

                elif hasattr(image_features, "last_hidden_state"):
                    image_features = image_features.last_hidden_state[:, 0, :]

                else:
                    raise TypeError(
                        "Unexpected image embedding output: "
                        f"{type(image_features)}"
                    )

            if not isinstance(text_features, torch.Tensor):

                if hasattr(text_features, "pooler_output"):
                    text_features = text_features.pooler_output

                elif hasattr(text_features, "text_embeds"):
                    text_features = text_features.text_embeds

                elif hasattr(text_features, "last_hidden_state"):
                    text_features = text_features.last_hidden_state[:, 0, :]

                else:
                    raise TypeError(
                        "Unexpected text embedding output: "
                        f"{type(text_features)}"
                    )

            # ------------------------------------------------
            # NORMALIZE EMBEDDINGS
            # ------------------------------------------------

            image_features = F.normalize(
                image_features,
                p=2,
                dim=-1
            )

            text_features = F.normalize(
                text_features,
                p=2,
                dim=-1
            )

            # ------------------------------------------------
            # STORE EMBEDDINGS
            # ------------------------------------------------

            all_image_embeddings.append(
                image_features.cpu()
            )

            all_text_embeddings.append(
                text_features.cpu()
            )

            # Close images to release memory.
            for image in images:
                image.close()

    # --------------------------------------------------------
    # COMBINE BATCHES
    # --------------------------------------------------------

    image_embeddings = torch.cat(
        all_image_embeddings,
        dim=0
    )

    text_embeddings = torch.cat(
        all_text_embeddings,
        dim=0
    )

    return image_embeddings, text_embeddings


# ------------------------------------------------------------
# GENERATE TRAINING EMBEDDINGS
# ------------------------------------------------------------

print("\nGenerating training embeddings...")

train_image_emb, train_text_emb = generate_embeddings(
    dataframe=train_df,
    batch_size=BATCH_SIZE,
    model=clip_model,
    processor=processor,
    device=DEVICE
)


# ------------------------------------------------------------
# GENERATE VALIDATION EMBEDDINGS
# ------------------------------------------------------------

print("\nGenerating validation embeddings...")

val_image_emb, val_text_emb = generate_embeddings(
    dataframe=val_df,
    batch_size=BATCH_SIZE,
    model=clip_model,
    processor=processor,
    device=DEVICE
)


# ------------------------------------------------------------
# 9.5 GENERATE TEST EMBEDDINGS
# ------------------------------------------------------------

print("\nGenerating test embeddings...")

test_image_emb, test_text_emb = generate_embeddings(
    dataframe=test_df,
    batch_size=BATCH_SIZE,
    model=clip_model,
    processor=processor,
    device=DEVICE
)


# ------------------------------------------------------------
# 9.6 VERIFY EMBEDDINGS
# ------------------------------------------------------------

print("\nEmbedding generation completed.")

print("\nEmbedding dimensions:")

print("Training images:", train_image_emb.shape)
print("Training texts:", train_text_emb.shape)

print("Validation images:", val_image_emb.shape)
print("Validation texts:", val_text_emb.shape)

print("Test images:", test_image_emb.shape)
print("Test texts:", test_text_emb.shape)


# ------------------------------------------------------------
# 9.7 VERIFY NORMALIZATION
# ------------------------------------------------------------

print("\nChecking embedding normalization:")

print(
    "Training image norms:",
    torch.norm(train_image_emb, dim=1)[:5]
)

print(
    "Training text norms:",
    torch.norm(train_text_emb, dim=1)[:5]
)


# ------------------------------------------------------------
# 9.8 VERIFY EMBEDDING COUNTS
# ------------------------------------------------------------

assert len(train_image_emb) == len(train_df)
assert len(train_text_emb) == len(train_df)

assert len(val_image_emb) == len(val_df)
assert len(val_text_emb) == len(val_df)

assert len(test_image_emb) == len(test_df)
assert len(test_text_emb) == len(test_df)

print("\nAll embedding counts match their respective datasets.")



Generating training embeddings...


Generating embeddings:   0%|          | 0/90 [00:00<?, ?it/s]


Generating validation embeddings...


Generating embeddings:   0%|          | 0/21 [00:00<?, ?it/s]


Generating test embeddings...


Generating embeddings:   0%|          | 0/15 [00:00<?, ?it/s]


Embedding generation completed.

Embedding dimensions:
Training images: torch.Size([1431, 512])
Training texts: torch.Size([1431, 512])
Validation images: torch.Size([321, 512])
Validation texts: torch.Size([321, 512])
Test images: torch.Size([229, 512])
Test texts: torch.Size([229, 512])

Checking embedding normalization:
Training image norms: tensor([1., 1., 1., 1., 1.])
Training text norms: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000])

All embedding counts match their respective datasets.


In [15]:
# ============================================================
# 10. CALCULATE COSINE SIMILARITY
# ============================================================
def calculate_similarity(image_embeddings, text_embeddings):

    # Ensure inputs are PyTorch tensors.
    if not isinstance(image_embeddings, torch.Tensor):
        image_embeddings = torch.tensor(image_embeddings)

    if not isinstance(text_embeddings, torch.Tensor):
        text_embeddings = torch.tensor(text_embeddings)

    # Move tensors to CPU for similarity calculation.
    image_embeddings = image_embeddings.float().cpu()
    text_embeddings = text_embeddings.float().cpu()

    # Normalize embeddings.
    image_embeddings = F.normalize(
        image_embeddings,
        p=2,
        dim=1
    )

    text_embeddings = F.normalize(
        text_embeddings,
        p=2,
        dim=1
    )

    # Calculate pairwise cosine similarity.
    # Each image is compared with its corresponding alt text.
    similarity_scores = torch.sum(
        image_embeddings * text_embeddings,
        dim=1
    )

    return similarity_scores.numpy()


# ============================================================
# CALCULATE SIMILARITY FOR EACH DATA PARTITION
# ============================================================

print("\nCalculating training similarity...")

train_similarity = calculate_similarity(
    train_image_emb,
    train_text_emb
)

print("\nCalculating validation similarity...")

val_similarity = calculate_similarity(
    val_image_emb,
    val_text_emb
)

print("\nCalculating test similarity...")

test_similarity = calculate_similarity(
    test_image_emb,
    test_text_emb
)


# ============================================================
# VERIFY SIMILARITY SCORES
# ============================================================

print("\nSimilarity calculation completed.")

print("Training similarity shape:", train_similarity.shape)
print("Validation similarity shape:", val_similarity.shape)
print("Test similarity shape:", test_similarity.shape)

print("\nTraining similarity statistics:")

print("Minimum:", train_similarity.min())
print("Maximum:", train_similarity.max())
print("Mean:", train_similarity.mean())
print("Standard deviation:", train_similarity.std())

print("\nFirst 10 training similarity scores:")

print(train_similarity[:10])



Calculating training similarity...

Calculating validation similarity...

Calculating test similarity...

Similarity calculation completed.
Training similarity shape: (1431,)
Validation similarity shape: (321,)
Test similarity shape: (229,)

Training similarity statistics:
Minimum: 0.12900057
Maximum: 0.40237704
Mean: 0.23818816
Standard deviation: 0.04140373

First 10 training similarity scores:
[0.30783576 0.22082551 0.23009987 0.20034967 0.30336317 0.23911461
 0.2284284  0.21947834 0.21028218 0.29074675]


In [16]:
# ============================================================
# 11. SELECT BEST CLIP SIMILARITY THRESHOLD
# ============================================================

def find_best_threshold(y_true, scores):

    precision, recall, thresholds = precision_recall_curve(
        y_true,
        scores
    )

    # The last precision/recall values have no corresponding
    # threshold, so exclude them.

    precision = precision[:-1]
    recall = recall[:-1]

    f1_scores = (
        2 * precision * recall /
        (precision + recall + 1e-10)
    )

    best_idx = np.argmax(f1_scores)

    best_threshold = thresholds[best_idx]
    best_f1 = f1_scores[best_idx]

    return float(best_threshold), float(best_f1)


y_train = train_df["label"].to_numpy()
y_val = val_df["label"].to_numpy()
y_test = test_df["label"].to_numpy()

clip_threshold, validation_f1 = find_best_threshold(
    y_val,
    val_similarity
)

print("\nCLIP threshold selected using validation data:")
print("Threshold:", clip_threshold)
print("Validation F1:", validation_f1)


CLIP threshold selected using validation data:
Threshold: 0.23571069538593292
Validation F1: 0.8476821191555173


In [17]:
# ============================================================
# 12. TRAIN LOGISTIC REGRESSION BASELINE
# ============================================================

# ------------------------------------------------------------
# PREPARE ALTERNATIVE TEXT
# ------------------------------------------------------------

# Replace missing values with empty strings.

X_train_text = (
    train_df["alt_text"]
    .fillna("")
    .astype(str)
)

X_val_text = (
    val_df["alt_text"]
    .fillna("")
    .astype(str)
)

X_test_text = (
    test_df["alt_text"]
    .fillna("")
    .astype(str)
)


# ------------------------------------------------------------
# CREATE TF-IDF FEATURES
# ------------------------------------------------------------

tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    lowercase=True,
    strip_accents="unicode",
    sublinear_tf=True
)

# Fit only on training data.
X_train_lr = tfidf_vectorizer.fit_transform(
    X_train_text
)

# Transform validation and test data.
X_val_lr = tfidf_vectorizer.transform(
    X_val_text
)

X_test_lr = tfidf_vectorizer.transform(
    X_test_text
)


print("TF-IDF feature dimensions:")
print("Training:", X_train_lr.shape)
print("Validation:", X_val_lr.shape)
print("Testing:", X_test_lr.shape)


# ------------------------------------------------------------
# TRAIN LOGISTIC REGRESSION
# ------------------------------------------------------------

logistic_model = LogisticRegression(
    class_weight="balanced",
    random_state=SEED,
    max_iter=2000,
    C=1.0,
    solver="liblinear"
)

logistic_model.fit(
    X_train_lr,
    y_train
)


# ------------------------------------------------------------
# GENERATE PREDICTION PROBABILITIES
# ------------------------------------------------------------

lr_val_probabilities = logistic_model.predict_proba(
    X_val_lr
)[:, 1]

lr_test_probabilities = logistic_model.predict_proba(
    X_test_lr
)[:, 1]


# ------------------------------------------------------------
# SELECT THRESHOLD USING VALIDATION DATA
# ------------------------------------------------------------

lr_threshold, lr_validation_f1 = find_best_threshold(
    y_val,
    lr_val_probabilities
)


print("\nTF-IDF Logistic Regression results:")

print("Validation threshold:", lr_threshold)

print("Validation F1:", lr_validation_f1)


# ------------------------------------------------------------
# GENERATE FINAL TEST PREDICTIONS
# ------------------------------------------------------------

lr_test_predictions = (
    lr_test_probabilities >= lr_threshold
).astype(int)


print("\nTest predictions generated.")

print("Number of predictions:", len(lr_test_predictions))

print(
    "Predicted classes:",
    np.unique(
        lr_test_predictions,
        return_counts=True
    )
)

TF-IDF feature dimensions:
Training: (1431, 2449)
Validation: (321, 2449)
Testing: (229, 2449)

TF-IDF Logistic Regression results:
Validation threshold: 0.44673682954589106
Validation F1: 0.9968847351524927

Test predictions generated.
Number of predictions: 229
Predicted classes: (array([0, 1]), array([111, 118]))


In [18]:
# ============================================================
# 13. EVALUATION FUNCTION
# ============================================================

def evaluate_model(
    model_name,
    y_true,
    predictions,
    probabilities
):

    results = {
        "Model": model_name,

        "Accuracy": accuracy_score(
            y_true,
            predictions
        ),

        "Precision": precision_score(
            y_true,
            predictions,
            zero_division=0
        ),

        "Recall": recall_score(
            y_true,
            predictions,
            zero_division=0
        ),

        "F1 Score": f1_score(
            y_true,
            predictions,
            zero_division=0
        ),

        "ROC AUC": roc_auc_score(
            y_true,
            probabilities
        )
    }

    print("\n" + "=" * 55)
    print(model_name)
    print("=" * 55)

    for metric, value in results.items():

        if metric != "Model":
            print(f"{metric}: {value:.4f}")

    print("\nConfusion Matrix:")

    print(
        confusion_matrix(
            y_true,
            predictions,
            labels=[0, 1]
        )
    )

    print("\nClassification Report:")

    print(
        classification_report(
            y_true,
            predictions,
            target_names=[
                "Nonmeaningful",
                "Meaningful"
            ],
            zero_division=0
        )
    )

    return results


In [19]:
# ============================================================
# 14. TEST CLIP THRESHOLD MODEL
# ============================================================

clip_test_predictions = (
    test_similarity >= clip_threshold
).astype(int)

clip_results = evaluate_model(
    model_name="CLIP Similarity Threshold",
    y_true=y_test,
    predictions=clip_test_predictions,
    probabilities=test_similarity
)


CLIP Similarity Threshold
Accuracy: 0.8821
Precision: 0.8793
Recall: 0.8870
F1 Score: 0.8831
ROC AUC: 0.9429

Confusion Matrix:
[[100  14]
 [ 13 102]]

Classification Report:
               precision    recall  f1-score   support

Nonmeaningful       0.88      0.88      0.88       114
   Meaningful       0.88      0.89      0.88       115

     accuracy                           0.88       229
    macro avg       0.88      0.88      0.88       229
 weighted avg       0.88      0.88      0.88       229



In [20]:
# ============================================================
# 15. TEST LOGISTIC REGRESSION
# ============================================================

lr_test_predictions = (
    lr_test_probabilities >= lr_threshold
).astype(int)

lr_results = evaluate_model(
    model_name="Logistic Regression Baseline",
    y_true=y_test,
    predictions=lr_test_predictions,
    probabilities=lr_test_probabilities
)



Logistic Regression Baseline
Accuracy: 0.9782
Precision: 0.9661
Recall: 0.9913
F1 Score: 0.9785
ROC AUC: 0.9973

Confusion Matrix:
[[110   4]
 [  1 114]]

Classification Report:
               precision    recall  f1-score   support

Nonmeaningful       0.99      0.96      0.98       114
   Meaningful       0.97      0.99      0.98       115

     accuracy                           0.98       229
    macro avg       0.98      0.98      0.98       229
 weighted avg       0.98      0.98      0.98       229



In [21]:
# ============================================================
# 16. COMPARE MODELS
# ============================================================

comparison_df = pd.DataFrame([
    clip_results,
    lr_results
])

print("\n\nFINAL MODEL COMPARISON")
print("=" * 65)

print(
    comparison_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

comparison_df.to_csv(
    os.path.join(OUTPUT_DIR, "model_comparison.csv"),
    index=False
)



FINAL MODEL COMPARISON
                       Model  Accuracy  Precision  Recall  F1 Score  ROC AUC
   CLIP Similarity Threshold    0.8821     0.8793  0.8870    0.8831   0.9429
Logistic Regression Baseline    0.9782     0.9661  0.9913    0.9785   0.9973


In [22]:
# ============================================================
# 17. SAVE TEST PREDICTIONS
# ============================================================

test_predictions_df = test_df.copy()

test_predictions_df["clip_similarity"] = test_similarity

test_predictions_df["clip_prediction"] = (
    clip_test_predictions
)

test_predictions_df["logistic_probability"] = (
    lr_test_probabilities
)

test_predictions_df["logistic_prediction"] = (
    lr_test_predictions
)

test_predictions_df.to_csv(
    os.path.join(OUTPUT_DIR, "test_predictions.csv"),
    index=False
)

In [28]:
# ============================================================
# 18. SAVE TRAINED ARTIFACTS
# ============================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Saving trained artifacts to:", OUTPUT_DIR)


# ------------------------------------------------------------
# SAVE FINE-TUNED CLIP MODEL
# ------------------------------------------------------------

# Create a separate directory for the CLIP model.
CLIP_OUTPUT_DIR = os.path.join(
    OUTPUT_DIR,
    "trained_clip_model"
)

os.makedirs(CLIP_OUTPUT_DIR, exist_ok=True)

# Save fine-tuned CLIP architecture and model weights.
clip_model.save_pretrained(CLIP_OUTPUT_DIR)

# Save the corresponding processor and tokenizer.
processor.save_pretrained(CLIP_OUTPUT_DIR)

print("Fine-tuned CLIP model saved.")


# ------------------------------------------------------------
# SAVE CLIP MODEL WEIGHTS AS A .PT FILE
# ------------------------------------------------------------

CLIP_WEIGHTS_PATH = os.path.join(
    OUTPUT_DIR,
    "clip_model_weights.pt"
)

torch.save(
    clip_model.state_dict(),
    CLIP_WEIGHTS_PATH
)

print("CLIP model weights saved.")


# ------------------------------------------------------------
# SAVE LOGISTIC REGRESSION MODEL
# ------------------------------------------------------------

LR_MODEL_PATH = os.path.join(
    OUTPUT_DIR,
    "logistic_regression_model.pkl"
)

joblib.dump(
    logistic_model,
    LR_MODEL_PATH
)

print("Logistic regression model saved.")


# ------------------------------------------------------------
# SAVE TF-IDF VECTORIZER
# ------------------------------------------------------------

# Required to transform new alt-text examples before
# passing them into the trained logistic regression model.

TFIDF_PATH = os.path.join(
    OUTPUT_DIR,
    "tfidf_vectorizer.pkl"
)

joblib.dump(
    tfidf_vectorizer,
    TFIDF_PATH
)

print("TF-IDF vectorizer saved.")


# ------------------------------------------------------------
# SAVE MODEL CONFIGURATION
# ------------------------------------------------------------

model_config = {
    "clip_model": CLIP_MODEL_NAME,
    "clip_similarity_threshold": float(clip_threshold),
    "logistic_regression_threshold": float(lr_threshold),
    "random_seed": int(SEED),
    "similarity_metric": "cosine_similarity",
    "positive_class": "meaningful",
    "negative_class": "nonmeaningful",
    "classification_method": {
        "clip": "image_text_cosine_similarity",
        "logistic_regression": "tfidf_alt_text_classification"
    }
}

CONFIG_PATH = os.path.join(
    OUTPUT_DIR,
    "model_config.json"
)

with open(CONFIG_PATH, "w") as f:
    json.dump(
        model_config,
        f,
        indent=4
    )

print("Model configuration saved.")


# ------------------------------------------------------------
# SAVE EVALUATION RESULTS
# ------------------------------------------------------------

def make_json_serializable(value):

    if isinstance(value, dict):
        return {
            key: make_json_serializable(item)
            for key, item in value.items()
        }

    elif isinstance(value, (list, tuple)):
        return [
            make_json_serializable(item)
            for item in value
        ]

    elif isinstance(value, np.ndarray):
        return value.tolist()

    elif isinstance(value, (np.integer,)):
        return int(value)

    elif isinstance(value, (np.floating,)):
        return float(value)

    elif isinstance(value, (np.bool_,)):
        return bool(value)

    elif isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()

    else:
        return value


evaluation_results = {
    "clip_threshold_validation_f1": float(validation_f1),
    "logistic_validation_f1": float(lr_validation_f1),
    "test_results": [
        clip_results,
        lr_results
    ]
}

evaluation_results = make_json_serializable(
    evaluation_results
)

EVALUATION_PATH = os.path.join(
    OUTPUT_DIR,
    "evaluation_results.json"
)

with open(EVALUATION_PATH, "w") as f:
    json.dump(
        evaluation_results,
        f,
        indent=4
    )

print("Evaluation results saved.")


# ------------------------------------------------------------
# VERIFY SAVED ARTIFACTS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("SAVED MODEL ARTIFACTS")
print("=" * 60)

for root, dirs, files in os.walk(OUTPUT_DIR):

    level = root.replace(
        OUTPUT_DIR, ""
    ).count(os.sep)

    indent = "    " * level

    print(f"{indent}{os.path.basename(root)}/")

    sub_indent = "    " * (level + 1)

    for filename in files:
        filepath = os.path.join(root, filename)

        file_size_mb = os.path.getsize(filepath) / (1024 ** 2)

        print(
            f"{sub_indent}{filename} "
            f"({file_size_mb:.2f} MB)"
        )

print("\nAll trained artifacts saved successfully!")


Saving trained artifacts to: /content/drive/MyDrive/EEE6778_Semester_Project/clip_training_results


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned CLIP model saved.
CLIP model weights saved.
Logistic regression model saved.
TF-IDF vectorizer saved.
Model configuration saved.
Evaluation results saved.

SAVED MODEL ARTIFACTS
clip_training_results/
    clip_model_weights.pt (577.20 MB)
    logistic_regression_model.pkl (0.02 MB)
    tfidf_vectorizer.pkl (0.09 MB)
    model_config.json (0.00 MB)
    evaluation_results.json (0.00 MB)
    trained_clip_model/
        config.json (0.00 MB)
        model.safetensors (577.12 MB)
        tokenizer_config.json (0.00 MB)
        tokenizer.json (3.47 MB)
        processor_config.json (0.00 MB)

All trained artifacts saved successfully!


In [29]:
# ============================================================
# 19. SAVE CLIP EMBEDDINGS
# ============================================================

np.savez_compressed(
    os.path.join(OUTPUT_DIR, "clip_embeddings.npz"),

    train_image_embeddings=train_image_emb,
    train_text_embeddings=train_text_emb,

    validation_image_embeddings=val_image_emb,
    validation_text_embeddings=val_text_emb,

    test_image_embeddings=test_image_emb,
    test_text_embeddings=test_text_emb
)


In [30]:
# ============================================================
# 20. FINAL OUTPUT
# ============================================================

print("\n\nTRAINING AND EVALUATION COMPLETED!")
print("=" * 65)

print("\nCLIP threshold:", clip_threshold)
print("Logistic regression threshold:", lr_threshold)

print("\nResults saved to:")
print(OUTPUT_DIR)

print("\nGenerated files:")

for filename in sorted(os.listdir(OUTPUT_DIR)):
    print("-", filename)



TRAINING AND EVALUATION COMPLETED!

CLIP threshold: 0.23571069538593292
Logistic regression threshold: 0.44673682954589106

Results saved to:
/content/drive/MyDrive/EEE6778_Semester_Project/clip_training_results

Generated files:
- clip_embeddings.npz
- clip_model_weights.pt
- evaluation_results.json
- logistic_regression_model.pkl
- model_config.json
- tfidf_vectorizer.pkl
- trained_clip_model
